# MARS Mouse Pose — Interview Prep EDA

**Dataset:** The Mouse Action Recognition System (MARS) — pose annotation data  
**Source:** https://data.caltech.edu/records/j1ww1-mdc55  
**Files used:** `MARS_keypoints_top.json` (top-view, 21 MB) and `MARS_keypoints_front.json` (front-view, 28 MB)

### What this notebook does
1. Loads and explores the schema
2. Computes summary statistics
3. Visualises keypoint distributions and skeleton overlays
4. Frames every step as a **pipeline / MLOps decision** so you can talk about it in an interview


## 1 · Set Up Environment and Dependencies

In [ ]:
import json
import pathlib
import random
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE = pathlib.Path(r"c:\Users\mro84\Machine Vision")
DATA = BASE / "data"
OUT  = BASE / "data" / "eda_outputs"
OUT.mkdir(parents=True, exist_ok=True)

TOP_JSON   = DATA / "MARS_keypoints_top.json"
FRONT_JSON = DATA / "MARS_keypoints_front.json"

# ── Matplotlib style ──────────────────────────────────────────────────────────
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

print("Libraries loaded. Output folder:", OUT)


## 2 · Load Data and Explore Schema

In [ ]:
top_raw   = json.loads(TOP_JSON.read_text())
front_raw = json.loads(FRONT_JSON.read_text())

print(f"Top-view frames   : {len(top_raw):,}")
print(f"Front-view frames : {len(front_raw):,}")

sample = top_raw[0]
print("\n── First frame (top-view) ──────────────────")
print("  filename :", sample["filename"])
print("  resolution:", sample["width"], "×", sample["height"])
print("  labels   :", sample["labels"])
print("  mice     :", list(sample["coords"].keys()))
print("  black x  :", sample["coords"]["black"]["x"])
print("  black y  :", sample["coords"]["black"]["y"])


## 3 · Core Processing — Build a Flat DataFrame

**MLOps note:** This is the equivalent of a feature-extraction step. Converting nested JSON into a flat, typed table is the first concrete transformation. In production you'd version this artifact separately from the raw annotation JSON.

In [ ]:
def flatten_frames(records, view="top"):
    """Convert raw JSON records to a list of flat dicts, one row per mouse per frame."""
    rows = []
    kp_labels = records[0]["labels"][:7]   # 7 training keypoints (tail mid/end excluded)
    for rec in records:
        fname  = rec["filename"]
        w, h   = rec["width"], rec["height"]
        coords = rec["coords"]
        for mouse_id, kp_data in coords.items():
            xs = kp_data["x"]
            ys = kp_data["y"]
            row = {
                "filename" : fname,
                "view"     : view,
                "mouse"    : mouse_id,
                "img_w"    : w,
                "img_h"    : h,
            }
            for i, label in enumerate(kp_labels):
                row[f"x_{label.replace(' ', '_')}"] = xs[i]
                row[f"y_{label.replace(' ', '_')}"] = ys[i]
            rows.append(row)
    return rows, kp_labels

top_rows,   kp_labels_top   = flatten_frames(top_raw,   view="top")
front_rows, kp_labels_front = flatten_frames(front_raw, view="front")

print(f"Top rows   : {len(top_rows):,}  (= 15000 frames × 2 mice)")
print(f"Front rows : {len(front_rows):,}")
print(f"Keypoint columns (top) : {kp_labels_top}")
print("\nSample flat row:")
for k, v in list(top_rows[0].items())[:10]:
    print(f"  {k:30s} {v}")


## 4 · Validation Checks

**MLOps note:** These are the data-quality gates that belong in an automated CI pipeline every time a new annotation batch arrives.

In [ ]:
def validate(rows, view, img_w, img_h, kp_count=7):
    errors = []
    for i, row in enumerate(rows):
        x_vals = [row[k] for k in row if k.startswith("x_")]
        y_vals = [row[k] for k in row if k.startswith("y_")]
        if len(x_vals) != kp_count:
            errors.append(f"[{view}] row {i}: expected {kp_count} keypoints, got {len(x_vals)}")
        for x in x_vals:
            if not (0 <= x <= img_w):
                errors.append(f"[{view}] row {i}: x={x} out of bounds (0–{img_w})")
        for y in y_vals:
            if not (0 <= y <= img_h):
                errors.append(f"[{view}] row {i}: y={y} out of bounds (0–{img_h})")
    return errors

top_errors   = validate(top_rows,   "top",   img_w=1024, img_h=570)
front_errors = validate(front_rows, "front", img_w=1280, img_h=500, kp_count=7)

assert len(top_rows)   == 30_000, f"Expected 30000 top rows, got {len(top_rows)}"
assert len(front_rows) == 30_000, f"Expected 30000 front rows, got {len(front_rows)}"

print(f"Top   validation errors : {len(top_errors)}")
print(f"Front validation errors : {len(front_errors)}")
if top_errors[:3]:
    print("Sample top errors:", top_errors[:3])
print("All checks passed.")


## 5 · Visualise Results

### 5a · Keypoint heatmaps (where do mice spend time in the cage?)

**MLOps note:** Spatial distribution shifts over experiment days = dataset drift. Monitoring this catches rig changes, lighting swaps, and animal condition changes before they corrupt model performance.

In [ ]:
# Extract nose-tip positions for each mouse as a proxy for location
def extract_xy(rows, mouse_id, kp="nose_tip"):
    xs = [r[f"x_{kp}"] for r in rows if r["mouse"] == mouse_id]
    ys = [r[f"y_{kp}"] for r in rows if r["mouse"] == mouse_id]
    return np.array(xs), np.array(ys)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
MICE   = ["black", "white"]
COLORS = ["#222222", "#cccccc"]
EDGE   = ["white",  "black"]

for ax, (mouse, color, ec) in zip(axes, zip(MICE, COLORS, EDGE)):
    xs, ys = extract_xy(top_rows, mouse, "nose_tip")
    h = ax.hist2d(xs, ys, bins=60, cmap="hot", range=[[0, 1024], [0, 570]])
    ax.set_title(f"Nose-tip heatmap — {mouse} mouse (top view)", fontsize=11)
    ax.set_xlabel("x (pixels)")
    ax.set_ylabel("y (pixels)")
    ax.set_xlim(0, 1024)
    ax.set_ylim(570, 0)   # flip y so image-space is correct
    plt.colorbar(h[3], ax=ax, label="frame count")

plt.tight_layout()
plt.savefig(OUT / "nose_heatmaps_top.png", dpi=120)
plt.show()
print("Saved:", OUT / "nose_heatmaps_top.png")


### 5b · Skeleton overlays — visualise raw pose without original images

**MLOps note:** Rendering skeleton overlays from predicted vs ground-truth keypoints is how you debug a pose model. Being able to do this without the raw frames is valuable for remote review workflows.

In [ ]:
# Skeleton connectivity: pairs of keypoint indices to draw edges between
# Labels: nose_tip(0) r_ear(1) l_ear(2) neck(3) r_side(4) l_side(5) tail_base(6)
SKELETON = [(0, 1), (0, 2), (1, 3), (2, 3), (3, 4), (3, 5), (4, 6), (5, 6)]
KP_NAMES = [l.replace(" ", "_") for l in kp_labels_top]

def draw_skeleton(ax, row, color, alpha=0.85):
    xs = np.array([row[f"x_{k}"] for k in KP_NAMES])
    ys = np.array([row[f"y_{k}"] for k in KP_NAMES])
    for a, b in SKELETON:
        ax.plot([xs[a], xs[b]], [ys[a], ys[b]], color=color, lw=1.6, alpha=alpha)
    ax.scatter(xs, ys, s=18, color=color, zorder=5, edgecolors="white", linewidths=0.4)

# Sample 12 random frames and show both mice
sample_indices = random.sample(range(len(top_raw)), 12)
fig, axes = plt.subplots(3, 4, figsize=(16, 10),
                         subplot_kw={"aspect": "equal"})
axes = axes.flatten()

for ax, idx in zip(axes, sample_indices):
    frame = top_raw[idx]
    w, h  = frame["width"], frame["height"]
    ax.set_facecolor("#1e1e2e")
    ax.set_xlim(0, w); ax.set_ylim(h, 0)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(frame["filename"].replace("MARS_top_", ""), fontsize=7, color="#cdd6f4")

    for view_color, mouse_id in [("#f38ba8", "black"), ("#89dceb", "white")]:
        row_dict = {
            "mouse": mouse_id,
            **{f"x_{k}": frame["coords"][mouse_id]["x"][i] for i, k in enumerate(KP_NAMES)},
            **{f"y_{k}": frame["coords"][mouse_id]["y"][i] for i, k in enumerate(KP_NAMES)},
        }
        draw_skeleton(ax, row_dict, color=view_color)

legend_patches = [
    mpatches.Patch(color="#f38ba8", label="black mouse"),
    mpatches.Patch(color="#89dceb", label="white mouse"),
]
fig.legend(handles=legend_patches, loc="lower center", ncol=2, fontsize=9,
           frameon=False, labelcolor="white")
fig.patch.set_facecolor("#181825")
plt.suptitle("MARS top-view — skeleton overlays (12 random frames)", color="#cdd6f4", fontsize=12)
plt.tight_layout(rect=[0, 0.04, 1, 0.97])
plt.savefig(OUT / "skeleton_overlays_top.png", dpi=130, facecolor=fig.get_facecolor())
plt.show()
print("Saved:", OUT / "skeleton_overlays_top.png")


### 5c · Inter-mouse distance over frames

**MLOps note:** Inter-mouse nose distance is one of the simplest behavior-relevant features derived from pose. Distribution shifts in this signal are an early-warning indicator of corrupted tracking or unusual experimental conditions. Monitoring it in production is free and extremely informative.

In [ ]:
def nose_distances(records):
    dists = []
    for rec in records:
        bx = rec["coords"]["black"]["x"][0]
        by = rec["coords"]["black"]["y"][0]
        wx = rec["coords"]["white"]["x"][0]
        wy = rec["coords"]["white"]["y"][0]
        dists.append(np.sqrt((bx - wx) ** 2 + (by - wy) ** 2))
    return np.array(dists)

top_dists = nose_distances(top_raw)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(top_dists, bins=80, color="#89b4fa", edgecolor="none")
axes[0].axvline(np.median(top_dists), color="#f38ba8", lw=1.5, label=f"median {np.median(top_dists):.0f}px")
axes[0].axvline(np.percentile(top_dists, 10), color="#a6e3a1", lw=1.2, ls="--", label=f"P10 {np.percentile(top_dists,10):.0f}px")
axes[0].set_xlabel("Nose-to-nose distance (pixels)")
axes[0].set_ylabel("Frames")
axes[0].set_title("Inter-mouse distance — top view")
axes[0].legend(fontsize=8)

axes[1].plot(top_dists[:500], lw=0.7, color="#89b4fa", alpha=0.8)
axes[1].set_xlabel("Frame index (first 500)")
axes[1].set_ylabel("Distance (pixels)")
axes[1].set_title("Distance over time — first 500 frames")

plt.tight_layout()
plt.savefig(OUT / "inter_mouse_distance.png", dpi=120)
plt.show()

print(f"median {np.median(top_dists):.1f}px  |  "
      f"P5 {np.percentile(top_dists,5):.1f}px  |  "
      f"P95 {np.percentile(top_dists,95):.1f}px")
print("Saved:", OUT / "inter_mouse_distance.png")


## 6 · Export Outputs

Saves a JSON summary of dataset stats and a small CSV sample — the kind of artifact you'd version alongside a model in an MLOps system.

In [ ]:
import csv

# ── JSON summary ─────────────────────────────────────────────────────────────
summary = {
    "dataset"                : "MARS pose annotation data v1.0",
    "source"                 : "https://data.caltech.edu/records/j1ww1-mdc55",
    "top_view": {
        "frames"             : len(top_raw),
        "rows_after_flatten" : len(top_rows),
        "resolution_w"       : 1024,
        "resolution_h"       : 570,
        "keypoints_per_mouse": 7,
        "inter_mouse_dist_median_px": round(float(np.median(top_dists)), 2),
        "inter_mouse_dist_p5_px"   : round(float(np.percentile(top_dists, 5)), 2),
        "inter_mouse_dist_p95_px"  : round(float(np.percentile(top_dists, 95)), 2),
    },
    "front_view": {
        "frames"             : len(front_raw),
        "rows_after_flatten" : len(front_rows),
        "resolution_w"       : 1280,
        "resolution_h"       : 500,
        "keypoints_per_mouse": 7,
    },
    "pipeline_notes": [
        "Annotations are median of 5 AMT workers per keypoint — consensus label strategy worth discussing.",
        "Tail mid and end were collected but excluded from training set — label versioning decision.",
        "Identity tracked by coat colour (black vs white) — simple but fragile in lighting changes.",
        "15k frames sampled from 64 videos — discuss sampling strategy and temporal independence.",
    ]
}

summary_path = OUT / "dataset_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print("Wrote:", summary_path)

# ── CSV sample (first 200 rows of top-view flat table) ───────────────────────
csv_path = OUT / "top_keypoints_sample.csv"
fieldnames = list(top_rows[0].keys())
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(top_rows[:200])
print("Wrote:", csv_path)

print("\n── Export complete ──")
for p in sorted(OUT.iterdir()):
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:40s}  {size_kb:7.1f} KB")


## 7 · Baseline Feature Model (Interview Demo)

This section creates a **practical baseline** that mimics a behavior model workflow:
- engineer frame-level features from pose keypoints
- create a lightweight proxy target (`is_close_interaction`)
- train/evaluate a simple classifier
- export model metrics and scored samples

> Note: this is a demo baseline for interview preparation, not a scientific behavior label.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Sort frames by filename to get stable sequence order
top_sorted = sorted(top_raw, key=lambda r: r['filename'])

def frame_features(records):
    rows = []
    prev = None
    for i, rec in enumerate(records):
        b = rec['coords']['black']
        w = rec['coords']['white']

        # Key points used (index 0=nose tip, 3=neck, 6=tail base)
        b_nose = np.array([b['x'][0], b['y'][0]], dtype=float)
        w_nose = np.array([w['x'][0], w['y'][0]], dtype=float)
        b_neck = np.array([b['x'][3], b['y'][3]], dtype=float)
        w_neck = np.array([w['x'][3], w['y'][3]], dtype=float)
        b_tail = np.array([b['x'][6], b['y'][6]], dtype=float)
        w_tail = np.array([w['x'][6], w['y'][6]], dtype=float)

        nose_dist = float(np.linalg.norm(b_nose - w_nose))
        neck_dist = float(np.linalg.norm(b_neck - w_neck))
        tail_dist = float(np.linalg.norm(b_tail - w_tail))

        b_body_len = float(np.linalg.norm(b_nose - b_tail))
        w_body_len = float(np.linalg.norm(w_nose - w_tail))

        if prev is None:
            b_speed = 0.0
            w_speed = 0.0
            rel_speed = 0.0
        else:
            b_speed = float(np.linalg.norm(b_nose - prev['b_nose']))
            w_speed = float(np.linalg.norm(w_nose - prev['w_nose']))
            rel_speed = float(abs(nose_dist - prev['nose_dist']))

        rows.append({
            'frame_idx': i,
            'filename': rec['filename'],
            'nose_dist': nose_dist,
            'neck_dist': neck_dist,
            'tail_dist': tail_dist,
            'b_body_len': b_body_len,
            'w_body_len': w_body_len,
            'b_nose_speed': b_speed,
            'w_nose_speed': w_speed,
            'relative_dist_change': rel_speed,
        })

        prev = {'b_nose': b_nose, 'w_nose': w_nose, 'nose_dist': nose_dist}

    df = pd.DataFrame(rows)

    # Proxy target: closer-than-usual social proximity
    threshold = float(df['nose_dist'].quantile(0.25))
    df['is_close_interaction'] = (df['nose_dist'] <= threshold).astype(int)
    return df, threshold

feat_df, close_thresh = frame_features(top_sorted)
print('Feature table shape:', feat_df.shape)
print('Close interaction threshold (px):', round(close_thresh, 2))
feat_df.head(3)

In [ ]:
feature_cols = [
    'nose_dist',
    'neck_dist',
    'tail_dist',
    'b_body_len',
    'w_body_len',
    'b_nose_speed',
    'w_nose_speed',
    'relative_dist_change',
]

X = feat_df[feature_cols]
y = feat_df['is_close_interaction']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

clf = Pipeline([
    ('scale', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000, random_state=SEED))
])
clf.fit(X_train, y_train)

pred = clf.predict(X_test)
proba = clf.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, proba)

print('AUC:', round(float(auc), 4))
print('\nClassification report:\n')
print(classification_report(y_test, pred, digits=4))
print('Confusion matrix:\n', confusion_matrix(y_test, pred))

In [ ]:
# Visualise score distribution
plt.figure(figsize=(8, 4))
plt.hist(proba[y_test.values == 0], bins=50, alpha=0.7, label='not close')
plt.hist(proba[y_test.values == 1], bins=50, alpha=0.7, label='close')
plt.xlabel('Predicted probability of close interaction')
plt.ylabel('Count')
plt.title('Baseline model score distribution')
plt.legend()
plt.tight_layout()
plt.savefig(OUT / 'baseline_score_distribution.png', dpi=120)
plt.show()

# Export scored sample + model summary
scored = X_test.copy()
scored['y_true'] = y_test.values
scored['y_pred'] = pred
scored['y_proba_close'] = proba
scored = scored.reset_index(drop=True)

scored_path = OUT / 'baseline_scored_sample.csv'
scored.head(1000).to_csv(scored_path, index=False)

model_summary = {
    'model_type': 'logistic_regression',
    'features': feature_cols,
    'target': 'is_close_interaction (proxy)',
    'close_threshold_px': round(float(close_thresh), 3),
    'auc': round(float(auc), 5),
    'train_size': int(len(X_train)),
    'test_size': int(len(X_test)),
}
(OUT / 'baseline_model_summary.json').write_text(json.dumps(model_summary, indent=2))

print('Wrote:', OUT / 'baseline_score_distribution.png')
print('Wrote:', scored_path)
print('Wrote:', OUT / 'baseline_model_summary.json')